In [1]:
import os
import re
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [2]:
load_dotenv()

api_key = os.environ["GEMINI_API_KEY"]
print("API key loaded:", bool(api_key))

client = genai.Client(api_key=api_key)

API key loaded: True


In [3]:
def get_stock_price(symbol: str) -> dict:
    """Get the current stock price for a ticker symbol.
    Args:
        symbol: Stock ticker symbol, such as MSFT.
    """

    fake_prices = {
        "MSFT": 512.35,
        "AAPL": 230.10,
        "GOOG": 201.50,
    }

    symbol = symbol.upper()

    if symbol not in fake_prices:
        raise ValueError(f"Unknown stock symbol: {symbol}")

    return {
        "symbol": symbol,
        "price": fake_prices[symbol],
        "currency": "USD",
    }

In [4]:
def split_into_sentences(text):
    # Simple learning-oriented splitter
    return re.split(r"(?<=[.!?])\s+", text.strip())


# each chunk, which is a mini-paragraph, has max_chars, with overlap_sentences
def chunk_section(section_text, max_chars=500, overlap_sentences=1):
    sentences = split_into_sentences(section_text)
    chunks = []
    current = []

    for sentence in sentences:
        candidate = " ".join(current + [sentence])

        if len(candidate) <= max_chars:
            current.append(sentence)
        else:
            if current:
                chunks.append(" ".join(current))

            # Keep a little context from previous chunk
            if overlap_sentences > 0 and current:
                current = current[-overlap_sentences:]
            else:
                current = []

            current.append(sentence)

    # append the last current, list of sentences
    if current:
        chunks.append(" ".join(current))

    return chunks


def split_into_sections(text):
    sections = []
    current_section = "Introduction"
    current_lines = []

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("#"):
            if current_lines:
                sections.append(
                    {"section": current_section, "text": " ".join(current_lines)}
                )

            current_section = line.lstrip("#").strip()

            current_lines = []

        elif line:
            current_lines.append(line)

    if current_lines:
        sections.append({"section": current_section, "text": " ".join(current_lines)})

    return sections


def chunk_document(document, max_chars=500, overlap_sentences=1):
    chunks = []
    chunk_index = 0
    sections = split_into_sections(document["text"])

    for section in sections:
        section_chunks = chunk_section(
            section["text"], max_chars=max_chars, overlap_sentences=overlap_sentences
        )

        for text in section_chunks:
            chunk = {
                "chunk_id": (f"{document['document_id']}_{chunk_index:04d}"),
                "chunk_index": chunk_index,
                "document_id": document["document_id"],
                "title": document["title"],
                "category": document.get("category"),
                "source_path": document.get("source_path"),
                "section": section["section"],
                "text": text,
            }

            chunks.append(chunk)

            chunk_index += 1

    return chunks


def load_document(file_path, document_id, title, category=None):
    path = Path(file_path)

    with path.open("r", encoding="utf-8") as f:
        text = f.read()

    return {
        "document_id": document_id,
        "title": title,
        "category": category,
        "source_path": str(path),
        "text": text,
    }


def embed_text(text):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
    )

    return response.embeddings[0].values


def embed_chunks(chunks):
    for chunk in chunks:
        chunk["embedding"] = embed_text(chunk["text"])

    return chunks


def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def search_chunks(query, chunks, top_k=3):
    query_embedding = embed_text(query)
    results = []

    for chunk in chunks:
        score = cosine_similarity(query_embedding, chunk["embedding"])
        results.append({"chunk": chunk, "score": score})

    results.sort(key=lambda result: result["score"], reverse=True)

    return results[:top_k]


document = load_document("travel_policy.txt", "id_1", "Travel Policy")
chunks = chunk_document(document)
embedded_chunks = embed_chunks(chunks)


def search_documents(query: str) -> str:
    """Search internal company documents for information relevant to a query."""
    results = search_chunks(query=query, chunks=embedded_chunks, top_k=1)

    documents = []
    for result in results:
        chunk = result["chunk"]
        documents.append(
            {
                "chunk_id": chunk["chunk_id"],
                "document_id": chunk["document_id"],
                "title": chunk["title"],
                "section": chunk["section"],
                "text": chunk["text"],
                "score": float(result["score"]),
            }
        )

    return documents

In [5]:
search_documents_declaration = {
    "name": "search_documents",
    "description": (
        "Search internal company documents for information relevant to a query."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": ("Search query describing the information needed."),
            }
        },
        "required": ["query"],
    },
}

In [6]:
tool_registry = {
    "get_stock_price": get_stock_price,
    "search_documents": search_documents,
}

In [7]:
def execute_function_call(function_call):
    tool_name = function_call.name
    tool = tool_registry.get(tool_name)
    args = dict(function_call.args)

    if tool is None:
        return {
            "status": "error",
            "error": f"Unknown tool: {tool_name}",
        }

    try:
        result = tool(**args)
        return {
            "status": "success",
            "result": result,
        }
    except (TypeError, ValueError, KeyError, OSError) as e:
        return {
            "status": "error",
            "error": str(e),
        }

In [8]:
def rerank_chunks(original_query, chunks):
    query_embedding = embed_text(original_query)
    reranked_chunks = []

    for chunk in chunks:
        chunk_embedding = embed_text(chunk["text"])
        rerank_score = cosine_similarity(
            query_embedding,
            chunk_embedding,
        )

        reranked_chunk = chunk.copy()
        reranked_chunk["rerank_score"] = rerank_score
        reranked_chunks.append(reranked_chunk)

    reranked_chunks.sort(
        key=lambda x: x["rerank_score"],
        reverse=True,
    )

    return reranked_chunks

In [9]:
def build_retrieval_context(original_query, retrieved_chunks, max_chars=4000):
    seen = set()
    unique_chunks = []

    for chunk in retrieved_chunks:
        chunk_id = chunk["chunk_id"]

        if chunk_id in seen:
            continue

        seen.add(chunk_id)
        unique_chunks.append(chunk)

    reranked_chunks = rerank_chunks(
        original_query,
        unique_chunks,
    )

    selected_chunks = []
    used_chars = 0
    for chunk in reranked_chunks:
        chunk_size = len(chunk["text"])
        if used_chars + chunk_size > max_chars:
            continue

        selected_chunks.append(chunk)
        used_chars += chunk_size

    return selected_chunks

In [10]:
def format_retrieval_context(context_chunks):
    sections = []
    for chunk in context_chunks:
        sections.append(
            f"""SOURCE: {chunk["chunk_id"]}
            Document: {chunk["title"]}
            Section: {chunk["section"]} 
            Evidence:{chunk["text"]}"""
        )

    return "\n\n".join(sections)

In [14]:
def generate_grounded_answer(user_query, context_chunks):
    evidence = format_retrieval_context(context_chunks)
    prompt = f"""
Answer the user's question using only the evidence provided below.
Rules:
- Do not use information that is not supported by the evidence.
- Cite factual claims using the supporting SOURCE id in square brackets.
- If the evidence does not establish part of the answer, explicitly say so.
- Do not infer or invent missing company policy.

USER QUESTION:
{user_query}

EVIDENCE:
{evidence}
"""
    response = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt,
        config=types.GenerateContentConfig(
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True
            ),
        ),
    )

    return response.text

In [18]:
def run_agent(user_query, retrieval_max_steps=3, max_steps=10):
    contents = [types.Content(role="user", parts=[types.Part(text=user_query)])]

    state = {
        "contents": contents,
        "step": 0,
        "max_steps": max_steps,
        "retrieval_step": 0,
        "retrieval_max_steps": retrieval_max_steps,
        "retrieved_chunks": [],
        "status": "running",
    }

    print(f"Step {state['step']}")
    print(f"User query: {user_query}")

    while state["status"] == "running" and state["step"] < state["max_steps"]:
        response = client.models.generate_content(
            model="gemini-flash-lite-latest",
            contents=state["contents"],
            config=types.GenerateContentConfig(
                tools=tool_registry.values(),
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True
                ),
            ),
        )

        state["step"] += 1

        model_content = response.candidates[0].content
        state["contents"].append(model_content)

        function_calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call is not None
        ]

        # If there is no tool to run, return the text of the last response
        if not function_calls:
            state["status"] = "success"

            # No document retrieval happened.
            # The agent's direct answer is sufficient.
            if not state["retrieved_chunks"]:
                return response.text

            context_chunks = build_retrieval_context(
                user_query,
                state["retrieved_chunks"],
                max_chars=700
            )

            grounded_answer = generate_grounded_answer(user_query, context_chunks)
            return grounded_answer

        print(f"Step {state['step']}")

        function_response_parts = []

        for function_call in function_calls:
            if function_call.name == "search_documents" and state["retrieval_step"] >= state["retrieval_max_steps"]:
                result = {
                    "error": "retrieval_limit_reached",
                    "message": (
                        "Document retrieval limit reached. "
                        "No additional searches are allowed. "
                        "Use the evidence already retrieved to answer the user, "
                        "and clearly indicate anything that cannot be established "
                        "from the available evidence."
                    ),
                }
                print(f"result: {result}")
            else:
                try:
                    if function_call.name == "search_documents":
                        state["retrieval_step"] += 1

                    result = execute_function_call(function_call)

                    if function_call.name == "search_documents":
                        for chunk in result["result"]:
                            state['retrieved_chunks'].append(chunk)
                except Exception as e:
                    result = {
                        "error": "retrieval_failed",
                        "message": str(e),
                    }

            print(f"tool_name: {function_call.name}")
            print(f"args: {function_call.args}")
            print(f"result: {result}")

            function_response_parts.append(
                types.Part.from_function_response(
                    name=function_call.name,
                    response={"result": result},
                )
            )

        state["contents"].append(
            types.Content(role="user", parts=function_response_parts)
        )

    state["status"] = "failed"
    raise RuntimeError(f"Agent exceeded maximum number of steps ({max_steps})")

In [19]:
original_query = "I'm traveling to Tokyo for work. " + \
        "How much can I spend on my hotel and meals, " + \
        "and what receipts do I need to keep?"

result = run_agent(original_query)
print(result)

Step 0
User query: I'm traveling to Tokyo for work. How much can I spend on my hotel and meals, and what receipts do I need to keep?
Step 1
tool_name: search_documents
args: {'query': 'Tokyo hotel meals per diem travel policy receipts'}
result: {'status': 'success', 'result': [{'chunk_id': 'id_1_0003', 'document_id': 'id_1', 'title': 'Travel Policy', 'section': 'Meals', 'text': 'Employees may claim reasonable meal expenses incurred during approved business travel. The standard meal reimbursement limit is $80 per day for international travel and $65 per day for domestic travel. Alcoholic beverages are not reimbursable. Itemized receipts are required for individual meal expenses above $25.', 'score': 0.7520685919494928}]}
Step 2
tool_name: search_documents
args: {'query': 'hotel accommodation lodging limit Tokyo international travel rate'}
result: {'status': 'success', 'result': [{'chunk_id': 'id_1_0002', 'document_id': 'id_1', 'title': 'Travel Policy', 'section': 'Hotels', 'text': 'Empl